# Credit Risk & Loan Default Prediction: Exploratory Data Analysis (EDA)
### Ignosis FinTech Portfolio Project
**Dataset:** UCI Default of Credit Card Clients (Yeh & Lien, 2009)
**Objective:** Analyze borrower demographics, payment histories, and credit card balances to identify key drivers of default.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.loader import load_credit_card_data, TARGET_COL

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 30)

## 1. Data Ingestion & Schema Profiling

In [ ]:
df = load_credit_card_data()
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Target Distribution & Class Imbalance Analysis

In [ ]:
target_counts = df[TARGET_COL].value_counts()
target_pcts = df[TARGET_COL].value_counts(normalize=True)
print("Class Distribution:")
for k in [0, 1]:
    print(f"  Class {k}: {target_counts[k]} ({target_pcts[k]:.2%})")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x=TARGET_COL, data=df, palette=["#3B82F6", "#EF4444"], ax=ax)
ax.set_xticklabels(["Non-Default (0)", "Default (1)"])
ax.set_title("Loan Default Target Distribution (~22% Imbalance)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Delinquency Trajectory (Payment Status Analysis)
Evaluating default rates across historical payment delay categories (`PAY_1` to `PAY_6`).

In [ ]:
pay_default = df.groupby("PAY_1")[TARGET_COL].agg(["count", "mean"]).reset_index()
pay_default.columns = ["PAY_1 Status", "Applicant Count", "Default Rate"]
print(pay_default)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x="PAY_1 Status", y="Default Rate", data=pay_default, color="#F59E0B", ax=ax)
ax.axhline(df[TARGET_COL].mean(), color="red", linestyle="--", label="Population Default Rate (22.1%)")
ax.set_title("Default Rate by September Repayment Status (PAY_1)", fontsize=12, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Financial Capacity: Credit Limit vs. Default Rate

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(df[df[TARGET_COL] == 0]["LIMIT_BAL"], label="Non-Default", color="#3B82F6", fill=True, ax=ax)
sns.kdeplot(df[df[TARGET_COL] == 1]["LIMIT_BAL"], label="Default", color="#EF4444", fill=True, ax=ax)
ax.set_title("Credit Limit Distribution by Default Outcome", fontsize=12, fontweight="bold")
ax.set_xlabel("Credit Limit (LIMIT_BAL in NT$)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Key EDA Insights & Modeling Takeaways
1. **Class Imbalance:** 22.1% of clients defaulted. Accuracy is deceptive; primary evaluation metrics must be PR-AUC, ROC-AUC, Recall, and Expected Loss.
2. **Strongest Predictor:** Recent payment delinquency (`PAY_1`). Borrowers who are 2+ months delinquent default at rates over 50%.
3. **Feature Engineering Opportunities:** Domain transformations like credit utilization ratio (`BILL_AMT1 / LIMIT_BAL`), payment ratio, and delinquency counts (`months_delayed_count`) create powerful linear and nonlinear separation.